# 04 - Integrated Operational Silver
### Goal
- use `grid_events` as the base fact table
- aggregate weather and market prices to daily level before joining
- enrich with reference data
- write integrated silver Delta table at grid event granularity

In [0]:
%run ../01_setup/00_config

In [0]:
from pyspark.sql import functions as F

## Load silver and reference tables

In [0]:
market_prices_df        = spark.table(silver_market_prices_table)
weather_df              = spark.table(silver_weather_table)
grid_events_df          = spark.table(silver_grid_events_table)
asset_reference_df      = spark.table(silver_asset_reference_table)
event_type_reference_df = spark.table(silver_event_type_reference_table)
region_reference_df     = spark.table(silver_region_reference_table)

print(f"Market prices rows:        {market_prices_df.count()}")
print(f"Weather rows:              {weather_df.count()}")
print(f"Grid events rows:          {grid_events_df.count()}")
print(f"Asset reference rows:      {asset_reference_df.count()}")
print(f"Event type reference rows: {event_type_reference_df.count()}")
print(f"Region reference rows:     {region_reference_df.count()}")

## Aggregate weather and market prices to daily level

Both tables have multiple rows per `event_date`/`region`.
Aggregating before joining prevents row explosion.

In [0]:
market_prices_agg_df = (
    market_prices_df
    .groupBy("event_date", "region")
    .agg(
        F.round(F.avg("price_eur_mwh"), 2).alias("avg_price_eur_mwh"),
        F.round(F.sum("volume_mwh"), 2).alias("total_volume_mwh"),
        F.max("is_high_price").alias("has_high_price"),
        F.first("source_system").alias("market_source_system")
    )
)

weather_agg_df = (
    weather_df
    .groupBy("event_date", "region")
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temperature_c"),
        F.round(F.avg("wind_speed_kmh"), 2).alias("avg_wind_speed_kmh"),
        F.round(F.sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        F.max("is_severe_weather").alias("has_severe_weather"),
        F.first("wind_class").alias("wind_class")
    )
)

print(f"Market prices aggregated: {market_prices_agg_df.count()} rows")
print(f"Weather aggregated:       {weather_agg_df.count()} rows")

## Build integrated table

`grid_events` is the base fact. Each row represents one grid event enriched with daily market and weather context plus reference data.

In [0]:
integrated_df = (
    grid_events_df
    .join(
        asset_reference_df.select("asset_id", "asset_name", "asset_type"),
        on="asset_id",
        how="left"
    )
    .join(
        event_type_reference_df.select("event_type", "category"),
        on="event_type",
        how="left"
    )
    .join(
        region_reference_df.select("region", "country_code", "operating_zone"),
        on="region",
        how="left"
    )
    .join(
        weather_agg_df,
        on=["event_date", "region"],
        how="left"
    )
    .join(
        market_prices_agg_df,
        on=["event_date", "region"],
        how="left"
    )
)

display(integrated_df.limit(10))
print(f"Integrated rows: {integrated_df.count()}")
print(f"Expected:        same as grid_events ({grid_events_df.count()} rows)")

## Save silver integrated table

In [0]:
integrated_df.write.mode("overwrite").saveAsTable(silver_integrated_table)
print(f"Integrated silver table saved: {silver_integrated_table}")